In [1]:
import os

import numpy as np
import matplotlib.pyplot as plt

import obspy.taup
import obspy.taup.taup_create

%matplotlib inline 

In [2]:
from prem4derg import PREM, tabulate_model

## Tabulate the model

Because we have to

In [ ]:
# Get a table of PREM values every 10 km going inwards
# and dealing with the discontiuities 
table = tabulate_model(PREM, 20.0, outwards=False)
# Print the firs few depths
# note the discontiuities
print(np.record.pprint(table[0:7]))


NameError: name 'tabulate_model_inwards' is not defined

## Build an Obspy `TauPyModel`

And clean up the mess

In [ ]:
model_name = 'model'
tvel_filename = model_name + '.tvel'
taup_filename = model_name + '.npz'
f = open(tvel_filename, 'w')
f.write("P name\n")
f.write("S name\n")
for d, vp, vs, rho in zip(table.depth, table.vp, table.vs, table.density):
    f.write("{:10.3f} {:10.4f} {:10.4f} {:10.4f}\n".format(d, vp, vs, rho/1000.0))

f.close()

In [ ]:
# Look at the top of the tvel file (just to see what it looks like)
!head "$tvel_filename"

In [ ]:
# Build a taup model and store it in a numpy compressed file
obspy.taup.taup_create.build_taup_model(tvel_filename, output_folder='.', verbose=True)

In [ ]:
taup = obspy.taup.tau.TauPyModel('./'+taup_filename)

In [ ]:
# Clean up at the end (don't want to keep these file)
os.remove(tvel_filename)
os.remove(taup_filename)

## Create plot of raypaths and travel times

This now becomes easy

In [ ]:
fig, ax = plt.subplots(figsize=(10,10), subplot_kw={'projection':'polar'})

arrivals = taup.get_ray_paths(source_depth_in_km=500,
                               distance_in_degree=80,
                               phase_list=["P", "S", "PKP", 
                                           "PKIKP", "PKiKP",
                                           "S", "SKS"])

ax = arrivals.plot_rays(fig=fig, ax=ax, legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10,10), subplot_kw={'projection':'polar'})

arrivals = taup.get_ray_paths(source_depth_in_km=500,
                               distance_in_degree=150,
                               phase_list=["P", "S", "PKP", 
                                           "PKIKP", "PKiKP",
                                           "S", "SKS"])

ax = arrivals.plot_rays(fig=fig, ax=ax, legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax = obspy.taup.plot_travel_times(source_depth=10, phase_list=["P", "S", "PKP", 
                                                               "PKIKP", "PKiKP",
                                                               "S", "SKS"],
                       ax=ax, fig=fig, verbose=True, show=False)


plt.show()

In [ ]:
# NB - extracted from table VId
p_dists = np.array([23.0, 31.0, 41.0, 51.0, 61.0, 71.0, 81.0, 91.0])
p_times = np.array([263.57, 332.83, 415.35, 490.78, 559.81, 621.73, 676.86, 724.39])
p_error = np.array([0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax = obspy.taup.plot_travel_times(source_depth=550, phase_list=["P"],
                       ax=ax, fig=fig, verbose=True, show=False, legend=False,
                        max_degrees=100, npoints=50)

ax.errorbar(p_dists, p_times/60, yerr=p_error/60, fmt='r.')
ax.set_title("P calculated travel times and data")
plt.show()

In [ ]:
# NB - I've added 100 degrees to all PKP distances
#      I've a feeling this is a typo in table VIf...
pkp_dists = np.array([146.00, 148.00, 150.00, 152.00])
pkp_times = np.array([1180.50, 1186.11, 1191.25, 1195.87])
pkp_error = np.array([0.3, 0.3, 0.3, 0.3])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax = obspy.taup.plot_travel_times(source_depth=0, phase_list=["PKP"],
                       ax=ax, fig=fig, verbose=True, show=False, legend=False,
                        max_degrees=153, npoints=500)

ax.errorbar(pkp_dists, pkp_times/60, yerr=pkp_error/60, fmt='r.')
ax.set_title("PKPbc calculated travel times and data")
plt.show()